In [1]:

import pandas as pd
import json
import re
from collections import Counter
import nltk


nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:

df = pd.read_csv('music.csv', encoding='latin-1')


ratings = []
reviews = []

for json_str in df['X']:
    try:
        data = json.loads(json_str)
        rating = data.get('overall')
        

        review_text = None
        for key, value in data.items():
            if isinstance(value, str) and len(value) > 50:
                review_text = value
                break
        
        if rating is not None and review_text is not None:
            ratings.append(rating)
            reviews.append(review_text)
    except:
        continue

print(f"Loaded {len(reviews)} reviews")

Loaded 705 reviews


In [3]:

stop_words = set(stopwords.words('english'))
stop_words.update(['amp', 'br', 'http', 'https', 'com', 'www', 'quot', 'nbsp'])

def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

# Process all reviews
tokenized_reviews = []
for review in reviews:
    tokenized_reviews.append(clean_and_tokenize(review))

print(f"Tokenized {len(tokenized_reviews)} reviews")

Tokenized 705 reviews


In [4]:

word_counts = Counter()
for tokens in tokenized_reviews:
    word_counts.update(tokens)

print("Most common words in all reviews:")
print(word_counts.most_common(20))

Most common words in all reviews:
[('great', 135), ('guitar', 108), ('good', 88), ('price', 50), ('strings', 47), ('works', 37), ('better', 36), ('sound', 35), ('one', 34), ('quality', 34), ('like', 34), ('use', 32), ('pedal', 29), ('stand', 28), ('nice', 27), ('best', 27), ('acoustic', 26), ('well', 25), ('picks', 24), ('cable', 23)]


In [5]:

min_reviews = 5
word_scores = {}

for word, count in word_counts.items():
    total_rating = 0
    word_review_count = 0
    
    for i, tokens in enumerate(tokenized_reviews):
        if word in tokens:
            total_rating += ratings[i]
            word_review_count += 1
    
    if word_review_count >= min_reviews:
        word_scores[word] = total_rating / word_review_count

# Sort by sentiment score
sorted_words = sorted(word_scores.items(), key=lambda x: x[1], reverse=True)
print(f"Calculated scores for {len(word_scores)} words")

Calculated scores for 208 words


In [6]:

print("=" * 60)
print("TOP 20 POSITIVE WORDS (highest average rating)")
print("=" * 60)
for word, score in sorted_words[:20]:
    count = word_counts[word]
    print(f"{word:15} : {score:.2f} stars (appears {count} times)")

print("\n" + "=" * 60)
print("TOP 20 NEGATIVE WORDS (lowest average rating)")
print("=" * 60)
for word, score in sorted_words[-20:]:
    count = word_counts[word]
    print(f"{word:15} : {score:.2f} stars (appears {count} times)")

TOP 20 POSITIVE WORDS (highest average rating)
microphones     : 5.00 stars (appears 5 times)
awesome         : 5.00 stars (appears 9 times)
ever            : 5.00 stars (appears 5 times)
paul            : 5.00 stars (appears 6 times)
vintage         : 5.00 stars (appears 6 times)
phosphor        : 5.00 stars (appears 5 times)
bronze          : 5.00 stars (appears 5 times)
strumming       : 5.00 stars (appears 7 times)
elixir          : 5.00 stars (appears 5 times)
coating         : 5.00 stars (appears 5 times)
vocal           : 5.00 stars (appears 5 times)
display         : 5.00 stars (appears 5 times)
series          : 5.00 stars (appears 6 times)
behringer       : 5.00 stars (appears 7 times)
perfect         : 4.96 stars (appears 23 times)
made            : 4.86 stars (appears 7 times)
planet          : 4.85 stars (appears 13 times)
waves           : 4.85 stars (appears 13 times)
got             : 4.83 stars (appears 6 times)
must            : 4.83 stars (appears 6 times)

TOP 20 NE

In [7]:

results_df = pd.DataFrame(list(word_scores.items()), columns=['word', 'avg_rating'])
results_df['frequency'] = results_df['word'].map(word_counts)
results_df = results_df.sort_values('avg_rating', ascending=False)

print("Sample of word sentiment analysis (first 10 rows):")
results_df.head(10)

Sample of word sentiment analysis (first 10 rows):


,word,avg_rating,frequency
73,paul,5.0,6
75,vintage,5.0,6
69,ever,5.0,5
48,microphones,5.0,5
66,awesome,5.0,9
143,elixir,5.0,5
109,phosphor,5.0,5
196,behringer,5.0,7
144,coating,5.0,5
172,series,5.0,6


In [8]:

sound_words = ['sound', 'tone', 'quality', 'clear', 'crisp', 'muddy', 'bright', 'warm', 'harsh']

sound_results = results_df[results_df['word'].isin(sound_words)]
sound_results_sorted = sound_results.sort_values('avg_rating', ascending=False)

print("Sentiment analysis for sound-related words:")
print(sound_results_sorted)

Sentiment analysis for sound-related words:
        word  avg_rating  frequency
87      tone    4.666667         12
3      sound    4.529412         35
42   quality    4.500000         34
146   bright    4.400000          5


In [9]:

sound_words = ['sound', 'tone', 'quality', 'clear', 'crisp', 'muddy', 'bright', 'warm', 'harsh']

sound_results = results_df[results_df['word'].isin(sound_words)]
sound_results_sorted = sound_results.sort_values('avg_rating', ascending=False)

print("Sentiment analysis for sound-related words:")
print(sound_results_sorted)

Sentiment analysis for sound-related words:
        word  avg_rating  frequency
87      tone    4.666667         12
3      sound    4.529412         35
42   quality    4.500000         34
146   bright    4.400000          5


In [10]:

max_freq = results_df['frequency'].max()

if max_freq <= 100:
    bins = [0, 10, 50, max_freq]
elif max_freq <= 500:
    bins = [0, 10, 50, 100, max_freq]
elif max_freq <= 1000:
    bins = [0, 10, 50, 100, 500, max_freq]
else:
    bins = [0, 10, 50, 100, 500, 1000, max_freq]

bins = sorted(list(set(bins)))

print(f"Using bins: {bins}")
print(f"Max frequency: {max_freq}")

freq_bins = pd.cut(results_df['frequency'], bins=bins)
avg_sentiment_by_freq = results_df.groupby(freq_bins)['avg_rating'].mean()

print("\nAverage sentiment by word frequency:")
print(avg_sentiment_by_freq)

Using bins: [0, 10, 50, 100, np.int64(135)]
Max frequency: 135

Average sentiment by word frequency:
frequency
(0, 10]       4.332082
(10, 50]      4.358389
(50, 100]     4.129412
(100, 135]    4.528571
Name: avg_rating, dtype: float64


C:\Users\ashwa\AppData\Local\Temp\ipykernel_12924\3903454914.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  avg_sentiment_by_freq = results_df.groupby(freq_bins)['avg_rating'].mean()


In [11]:
print("=" * 60)
print("WORDS WITH PERFECT 5.0 STAR RATING:")
print("=" * 60)

if len(results_df[results_df['avg_rating'] == 5.0]) > 0:
    perfect_words = results_df[results_df['avg_rating'] == 5.0].sort_values('frequency', ascending=False).head(10)
    for _, row in perfect_words.iterrows():
        print(f"{row['word']:15} : appears {int(row['frequency'])} times")
else:
    print("No words with perfect 5.0 rating found")

print("\n" + "=" * 60)
print("WORDS WITH POOR 1.0 STAR RATING:")
print("=" * 60)

if len(results_df[results_df['avg_rating'] == 1.0]) > 0:
    poor_words = results_df[results_df['avg_rating'] == 1.0].sort_values('frequency', ascending=False).head(10)
    for _, row in poor_words.iterrows():
        print(f"{row['word']:15} : appears {int(row['frequency'])} times")
else:
    print("No words with 1.0 rating found")

WORDS WITH PERFECT 5.0 STAR RATING:
awesome         : appears 9 times
behringer       : appears 7 times
strumming       : appears 7 times
series          : appears 6 times
vintage         : appears 6 times
paul            : appears 6 times
elixir          : appears 5 times
microphones     : appears 5 times
ever            : appears 5 times
phosphor        : appears 5 times

WORDS WITH POOR 1.0 STAR RATING:
No words with 1.0 rating found


In [12]:

min_freq = 20  

positive_words = results_df[results_df['frequency'] >= min_freq].sort_values('avg_rating', ascending=False).head(10)
negative_words = results_df[results_df['frequency'] >= min_freq].sort_values('avg_rating', ascending=True).head(10)

print("=" * 60)
print("MOST POSITIVE WORDS (appear 20+ times):")
print("=" * 60)
for _, row in positive_words.iterrows():
    print(f"{row['word']:15} : {row['avg_rating']:.2f} stars ({int(row['frequency'])} times)")

print("\n" + "=" * 60)
print("MOST NEGATIVE WORDS (appear 20+ times):")
print("=" * 60)
for _, row in negative_words.iterrows():
    print(f"{row['word']:15} : {row['avg_rating']:.2f} stars ({int(row['frequency'])} times)")

MOST POSITIVE WORDS (appear 20+ times):
perfect         : 4.96 stars (23 times)
acoustic        : 4.77 stars (26 times)
excellent       : 4.76 stars (21 times)
product         : 4.70 stars (21 times)
pedal           : 4.68 stars (29 times)
cable           : 4.67 stars (23 times)
great           : 4.60 stars (135 times)
best            : 4.56 stars (27 times)
price           : 4.55 stars (50 times)
guitars         : 4.55 stars (23 times)

MOST NEGATIVE WORDS (appear 20+ times):
better          : 3.69 stars (36 times)
like            : 3.78 stars (34 times)
stand           : 3.89 stars (28 times)
use             : 3.97 stars (32 times)
good            : 4.13 stars (88 times)
picks           : 4.19 stars (24 times)
one             : 4.22 stars (34 times)
really          : 4.25 stars (21 times)
well            : 4.29 stars (25 times)
works           : 4.42 stars (37 times)


In [13]:

output_df = results_df.copy()
output_df['avg_rating'] = output_df['avg_rating'].round(2)
output_df = output_df.sort_values('avg_rating', ascending=False)



print("First 20 rows of complete analysis:")
output_df.head(20)

First 20 rows of complete analysis:


,word,avg_rating,frequency
73,paul,5.00,6
75,vintage,5.00,6
69,ever,5.00,5
48,microphones,5.00,5
66,awesome,5.00,9
143,elixir,5.00,5
109,phosphor,5.00,5
196,behringer,5.00,7
144,coating,5.00,5
172,series,5.00,6
